# Fine-tuning Qwen 3B pour la génération de JSON de produits de transport

Ce notebook permet d'entraîner un modèle Qwen 3B avec Unsloth pour convertir des descriptions en langage naturel en JSON structuré.

**Caractéristiques:**
- Modèle ultra-léger (Qwen 3B)
- Optimisé pour CPU avec GGUF
- Entraînement gratuit sur Google Colab
- Validation stricte avec reward modeling
- Conversion de descriptions → JSON sans erreurs

## 1. Installation des dépendances

In [ ]:
# Installation d'Unsloth et des dépendances
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps xformers trl peft accelerate bitsandbytes
!pip install -q datasets jsonschema

## 2. Imports et configuration

In [ ]:
import json
import torch
from datasets import Dataset
from unsloth import FastLanguageModel
from trl import SFTTrainer, DataCollatorForCompletionOnlyLM
from transformers import TrainingArguments
import jsonschema
from jsonschema import validate
import random

# Configuration
max_seq_length = 2048
dtype = None  # Auto-détection
load_in_4bit = True  # Pour économiser la mémoire

## 3. Chargement du schéma de produit de transport

In [ ]:
# Schéma JSON des produits de transport (simplifié pour l'exemple)
transport_schema = {
    "type": "object",
    "required": ["product_name", "characteristics"],
    "properties": {
        "product_name": {"type": "string"},
        "characteristics": {
            "type": "array",
            "items": {
                "type": "object",
                "required": ["number", "parameters"],
                "properties": {
                    "number": {"type": "integer"},
                    "parameters": {"type": "object"}
                }
            }
        }
    }
}

def validate_json(json_obj):
    """Valide un JSON selon le schéma"""
    try:
        validate(instance=json_obj, schema=transport_schema)
        return True
    except jsonschema.exceptions.ValidationError:
        return False

## 4. Génération du dataset d'entraînement

In [ ]:
# Exemples de données d'entraînement
training_data = [
    {
        "input": "Je veux créer un titre de transport valable 1 mois, avec 10 déplacements maximum, utilisable uniquement en bus et métro.",
        "output": {
            "product_name": "Abonnement mensuel 10 voyages Bus-Métro",
            "characteristics": [
                {
                    "number": 7,
                    "parameters": {
                        "7_01": 2,
                        "7_02": "M",
                        "7_03": 1,
                        "7_04": False,
                        "7_05": True
                    }
                },
                {
                    "number": 22,
                    "parameters": {
                        "22_01": 10,
                        "22_02": 10,
                        "22_03": False
                    }
                },
                {
                    "number": 14,
                    "parameters": {
                        "14_01": ["Bus urbain", "Métro"],
                        "14_02": "Autorisée"
                    }
                }
            ]
        }
    },
    {
        "input": "Créer un billet 24h valable sur tous les modes de transport, pour un groupe de 5 personnes maximum.",
        "output": {
            "product_name": "Pass 24h Groupe",
            "characteristics": [
                {
                    "number": 7,
                    "parameters": {
                        "7_01": 4,
                        "7_02": "H",
                        "7_03": 24,
                        "7_04": False,
                        "7_05": False
                    }
                },
                {
                    "number": 2,
                    "parameters": {
                        "2_01": 5
                    }
                }
            ]
        }
    },
    {
        "input": "Produit hebdomadaire rechargeable, valable uniquement en semaine de 9h à 17h, maximum 2 déplacements par jour.",
        "output": {
            "product_name": "Forfait hebdomadaire heures creuses",
            "characteristics": [
                {
                    "number": 7,
                    "parameters": {
                        "7_01": 2,
                        "7_02": "W",
                        "7_03": 1,
                        "7_04": True,
                        "7_05": True
                    }
                },
                {
                    "number": 9,
                    "parameters": {
                        "9_01": [
                            {"day": "Lundi-Vendredi", "start": "09:00", "end": "17:00"}
                        ]
                    }
                },
                {
                    "number": 10,
                    "parameters": {
                        "10_01": [
                            {"unit": "Jour", "count": 1, "max_trips": 2}
                        ]
                    }
                }
            ]
        }
    },
    {
        "input": "Titre annuel avec tacite reconduction, valable sur les lignes 1, 2 et 3, en première classe.",
        "output": {
            "product_name": "Abonnement annuel Premium Lignes 1-2-3",
            "characteristics": [
                {
                    "number": 7,
                    "parameters": {
                        "7_01": 2,
                        "7_02": "M",
                        "7_03": 12,
                        "7_04": True,
                        "7_05": True
                    }
                },
                {
                    "number": 102,
                    "parameters": {
                        "102_01": "Aucun",
                        "102_04": True
                    }
                },
                {
                    "number": 3,
                    "parameters": {
                        "3_01": ["Ligne 1", "Ligne 2", "Ligne 3"],
                        "3_02": "Autorisée"
                    }
                },
                {
                    "number": 58,
                    "parameters": {
                        "58_01": 1,
                        "58_02": True
                    }
                }
            ]
        }
    },
    {
        "input": "Carnet de 10 tickets, non rechargeable, utilisable sur tous les modes sauf le train.",
        "output": {
            "product_name": "Carnet 10 tickets multi-modal",
            "characteristics": [
                {
                    "number": 22,
                    "parameters": {
                        "22_01": 10,
                        "22_02": 10,
                        "22_03": False
                    }
                },
                {
                    "number": 14,
                    "parameters": {
                        "14_01": ["Train"],
                        "14_02": "Interdite"
                    }
                }
            ]
        }
    }
]

print(f"Dataset d'entraînement : {len(training_data)} exemples")

## 5. Préparation du dataset pour l'entraînement

In [ ]:
# Template de prompt
def format_prompt(input_text, output_json=None):
    """Formate le prompt pour l'entraînement"""
    prompt = f"""Vous êtes un assistant spécialisé dans la création de produits de transport. Convertissez la description suivante en JSON structuré.

### Description:
{input_text}

### JSON:"""
    
    if output_json is not None:
        prompt += f"\n{json.dumps(output_json, ensure_ascii=False, indent=2)}"
    
    return prompt

# Conversion en format dataset
formatted_data = []
for item in training_data:
    formatted_data.append({
        "text": format_prompt(item["input"], item["output"])
    })

dataset = Dataset.from_list(formatted_data)
print(f"Dataset créé avec {len(dataset)} exemples")
print("\nExemple de prompt formaté:")
print(dataset[0]["text"][:500] + "...")

## 6. Chargement du modèle Qwen 3B avec Unsloth

In [ ]:
# Chargement du modèle Qwen 3B
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-3B-Instruct",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

print("✓ Modèle Qwen 3B chargé avec succès")

## 7. Configuration LoRA pour le fine-tuning

In [ ]:
# Configuration LoRA (Low-Rank Adaptation)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,  # Rang LoRA
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

print("✓ Configuration LoRA appliquée")

## 8. Configuration de l'entraînement

In [ ]:
# Arguments d'entraînement
training_args = TrainingArguments(
    output_dir="./qwen3b-transport-finetuned",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_steps=10,
    max_steps=100,  # Augmenter pour un meilleur résultat
    learning_rate=2e-4,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=10,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="linear",
    seed=3407,
)

# Trainer
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    args=training_args,
)

print("✓ Trainer configuré")

## 9. Entraînement du modèle

In [ ]:
# Lancement de l'entraînement
print("🚀 Démarrage de l'entraînement...\n")
trainer_stats = trainer.train()
print("\n✓ Entraînement terminé !")

## 10. Test du modèle entraîné

In [ ]:
# Activation du mode inférence rapide
FastLanguageModel.for_inference(model)

# Test avec une nouvelle description
test_input = "Je veux un pass mensuel pour le métro et le tramway, valable du lundi au vendredi."

prompt = format_prompt(test_input)
inputs = tokenizer([prompt], return_tensors="pt").to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens=512,
    temperature=0.1,
    top_p=0.9,
    do_sample=True,
)

result = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("\n" + "="*50)
print("TEST DU MODÈLE")
print("="*50)
print(result)
print("="*50)

## 11. Validation avec reward scoring

In [ ]:
def extract_json_from_output(text):
    """Extrait le JSON de la sortie du modèle"""
    try:
        # Chercher le JSON entre les marqueurs
        start = text.find("### JSON:")
        if start != -1:
            json_text = text[start + len("### JSON:"):].strip()
            # Trouver le premier { et le dernier }
            first_brace = json_text.find("{")
            last_brace = json_text.rfind("}")
            if first_brace != -1 and last_brace != -1:
                json_text = json_text[first_brace:last_brace+1]
                return json.loads(json_text)
    except:
        pass
    return None

def calculate_reward(output_text):
    """Calcule le reward basé sur la validité du JSON"""
    json_obj = extract_json_from_output(output_text)
    
    if json_obj is None:
        return 0.0, "JSON invalide ou non trouvé"
    
    # Vérifications de base
    score = 0.0
    feedback = []
    
    # 1. Présence des champs requis
    if "product_name" in json_obj:
        score += 0.3
        feedback.append("✓ product_name présent")
    else:
        feedback.append("✗ product_name manquant")
    
    if "characteristics" in json_obj:
        score += 0.3
        feedback.append("✓ characteristics présent")
    else:
        feedback.append("✗ characteristics manquant")
    
    # 2. Validation du schéma
    if validate_json(json_obj):
        score += 0.4
        feedback.append("✓ Schéma JSON valide")
    else:
        feedback.append("✗ Schéma JSON invalide")
    
    return score, " | ".join(feedback)

# Test du reward
reward, feedback = calculate_reward(result)
print(f"\n📊 REWARD SCORE: {reward:.2f}/1.00")
print(f"📝 Feedback: {feedback}")

## 12. Export au format GGUF pour CPU

In [ ]:
# Sauvegarde du modèle LoRA
model.save_pretrained("qwen3b_transport_lora")
tokenizer.save_pretrained("qwen3b_transport_lora")
print("✓ Modèle LoRA sauvegardé")

# Fusion des poids LoRA avec le modèle de base
model.save_pretrained_merged(
    "qwen3b_transport_merged",
    tokenizer,
    save_method="merged_16bit",
)
print("✓ Modèle fusionné sauvegardé (16-bit)")

# Export GGUF pour utilisation CPU
model.save_pretrained_gguf(
    "qwen3b_transport_gguf",
    tokenizer,
    quantization_method="q4_k_m",  # Quantification 4-bit
)
print("✓ Modèle GGUF sauvegardé (Q4_K_M)")

# Export supplémentaire en Q8 pour meilleure qualité
model.save_pretrained_gguf(
    "qwen3b_transport_gguf",
    tokenizer,
    quantization_method="q8_0",
)
print("✓ Modèle GGUF sauvegardé (Q8_0)")

print("\n" + "="*50)
print("🎉 EXPORT TERMINÉ !")
print("="*50)
print("\nFichiers disponibles:")
print("  - qwen3b_transport_lora/ (adaptateurs LoRA)")
print("  - qwen3b_transport_merged/ (modèle complet 16-bit)")
print("  - qwen3b_transport_gguf/ (modèles GGUF pour CPU)")
print("\n💡 Utilisez les fichiers GGUF avec llama.cpp ou ollama pour inférence CPU")

## 13. Téléchargement des fichiers (Google Colab)

In [ ]:
# Compression des fichiers pour téléchargement
!apt-get install -y zip
!zip -r qwen3b_transport_gguf.zip qwen3b_transport_gguf/
!zip -r qwen3b_transport_lora.zip qwen3b_transport_lora/

print("✓ Fichiers compressés")
print("\n📥 Pour télécharger:")
print("  1. Allez dans le dossier Files à gauche")
print("  2. Clic droit sur les fichiers .zip")
print("  3. Sélectionnez 'Download'")

# Ou utilisez Google Drive (si connecté)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    
    !cp -r qwen3b_transport_gguf/ /content/drive/MyDrive/
    !cp -r qwen3b_transport_lora/ /content/drive/MyDrive/
    print("\n✓ Fichiers copiés vers Google Drive")
except:
    print("\n⚠ Google Drive non monté, utilisez le téléchargement manuel")